In [5]:
import re
import json
import pdfplumber
from pathlib import Path

def extract_novedades(pdf_path):
    # 1) Leer todo el texto del PDF
    full = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            txt = page.extract_text()
            if txt:
                full.append(txt)
    text = "\n".join(full)

    # 2) Capturar bloques
    arch_block = re.search(r"Proyectos Archivados:(.*?)(?:\(|Total)", text, re.S|re.I)
    ret_block  = re.search(r"Proyectos Retirados:(.*?)(?:\(|Total)", text, re.S|re.I)

    # 3) Extraer pares (número, año)
    arch_pairs = re.findall(r"(\d{1,4})/(\d{4})C", arch_block.group(1)) if arch_block else []
    ret_pairs  = re.findall(r"(\d{1,4})/(\d{4})C", ret_block.group(1))  if ret_block  else []

    return arch_pairs, ret_pairs

def build_table(arch_pairs, ret_pairs):
    # número de filas = la lista más larga
    n = max(len(arch_pairs), len(ret_pairs))
    rows = []
    for i in range(n):
        num_a, year_a = arch_pairs[i] if i < len(arch_pairs) else (None, None)
        num_r, year_r = ret_pairs[i]  if i < len(ret_pairs)  else (None, None)
        rows.append({
            "Proyectos Archivados": num_a,
            "Año Archivados"      : year_a,
            "Proyectos Retirados" : num_r,
            "Año Retirados"       : year_r
        })
    return rows

if __name__ == "__main__":
    pdf_file = r"C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\2018 2019 LEGISLATURA 2018-2019 (AGENDA LEGISLATIVA - CÁMARA) (1)_removed.pdf"  # ← ajusta la ruta a tu PDF
    arch_pairs, ret_pairs = extract_novedades(pdf_file)
    table = build_table(arch_pairs, ret_pairs)

    # 4) Guardar en JSON
    out_path = Path("novedades_v1.json")
    out_path.write_text(
        json.dumps(table, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    print(f"✅ Resultados guardados en {out_path.resolve()}")


CropBox missing from /Page, defaulting to MediaBox
CropBox missing from /Page, defaulting to MediaBox


✅ Resultados guardados en C:\Users\juans\Documents\proarchitecg\Model-Extract-information\extract\modelos\team\novedades_v1.json
